In [1]:
"""
v3-Final: Stylized Re-Augmentation
===================================
Generates 8 stylized variants per training image to simulate inter-patient
scope/illumination variance. Saves to disk for v3-final main pipeline.

Run ONCE before bladder-v3-final.py.
Runtime: ~15 minutes on Kaggle T4 (CPU-bound).
"""

import os
import re
import sys
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

# ─── CONFIG ───────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    DATASET_ROOT = '/kaggle/input/datasets/vrajpatel25/ebt-22'
    # Locate dynamically
    for root, dirs, _ in os.walk('/kaggle/input'):
        if 'EndoscopicBladderTissue' in dirs:
            DATASET_ROOT = root
            break
    ORIG_DATA_DIR = os.path.join(DATASET_ROOT, 'EndoscopicBladderTissue')
    if not os.path.isdir(ORIG_DATA_DIR):
        # Search recursively
        for root, dirs, _ in os.walk('/kaggle/input'):
            if os.path.basename(root) == 'EndoscopicBladderTissue':
                ORIG_DATA_DIR = root
                break
    
    # Annotations
    ANNOTATIONS_CSV = None
    for root, _, files in os.walk('/kaggle/input'):
        if 'annotations_fixed.csv' in files:
            ANNOTATIONS_CSV = os.path.join(root, 'annotations_fixed.csv')
            break
    
    OUTPUT_DIR = '/kaggle/working/v3_augmented'
    MANIFEST_OUT = '/kaggle/working/v3_augmented_manifest.csv'
else:
    script_dir = Path(__file__).resolve().parent
    workspace = script_dir.parent
    ORIG_DATA_DIR = str(workspace / 'EndoscopicBladderTissue')
    ANNOTATIONS_CSV = str(workspace / 'EndoscopicBladderTissue' / 'annotations_fixed.csv')
    OUTPUT_DIR = str(workspace / 'v3_augmented')
    MANIFEST_OUT = str(workspace / 'v3_augmented_manifest.csv')

TRAIN_PATIENTS = [4, 5, 7, 8, 10, 13, 14, 16, 21, 22, 23, 24, 25]
N_VARIANTS = 8  # 8 augmentations per train image
SEED = 42

random.seed(SEED)
np.random.seed(SEED)


def extract_patient_id(filename):
    for pat in [r'case_(\d+)', r'cys_case_(\d+)']:
        m = re.search(pat, str(filename))
        if m:
            return int(m.group(1))
    return -1


# ─── AUGMENTATION OPERATIONS ──────────────────────────────────
def random_hue_sat_shift(img):
    """Random hue and saturation shift (simulates different scope settings)."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    h_shift = random.uniform(-25, 25)
    s_mult = random.uniform(0.55, 1.45)
    v_mult = random.uniform(0.65, 1.35)
    hsv[:, :, 0] = (hsv[:, :, 0] + h_shift) % 180
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * s_mult, 0, 255)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * v_mult, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


def random_gamma(img):
    """Gamma correction (simulates different illumination)."""
    gamma = random.uniform(0.65, 1.45)
    inv = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)]).astype(np.uint8)
    return cv2.LUT(img, table)


def random_brightness_contrast(img):
    """Brightness + contrast jitter."""
    alpha = random.uniform(0.75, 1.30)  # contrast
    beta = random.uniform(-30, 30)       # brightness
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)


def random_motion_blur(img):
    """Motion blur (simulates scope movement)."""
    if random.random() > 0.4:
        return img
    ksize = random.choice([3, 5, 7])
    angle = random.uniform(0, 180)
    kernel = np.zeros((ksize, ksize))
    kernel[ksize // 2, :] = np.ones(ksize) / ksize
    M = cv2.getRotationMatrix2D((ksize / 2, ksize / 2), angle, 1)
    kernel = cv2.warpAffine(kernel, M, (ksize, ksize))
    kernel = kernel / (kernel.sum() + 1e-8)
    return cv2.filter2D(img, -1, kernel)


def random_jpeg_compression(img):
    """JPEG compression artifacts."""
    if random.random() > 0.5:
        return img
    quality = random.randint(35, 80)
    enc_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encoded = cv2.imencode('.jpg', img, enc_param)
    return cv2.imdecode(encoded, cv2.IMREAD_COLOR)


def vignette(img):
    """Scope-vignetting simulation."""
    if random.random() > 0.35:
        return img
    h, w = img.shape[:2]
    cx, cy = w / 2 + random.uniform(-w * 0.08, w * 0.08), h / 2 + random.uniform(-h * 0.08, h * 0.08)
    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
    max_dist = np.sqrt((w / 2) ** 2 + (h / 2) ** 2)
    strength = random.uniform(0.20, 0.55)
    mask = 1 - strength * (dist / max_dist) ** 2
    mask = np.clip(mask, 0, 1)
    out = img.astype(np.float32)
    for c in range(3):
        out[:, :, c] *= mask
    return np.clip(out, 0, 255).astype(np.uint8)


def random_geometric(img):
    """Geometric augmentation (flip + small rotation)."""
    if random.random() > 0.5:
        img = cv2.flip(img, 1)
    if random.random() > 0.5:
        img = cv2.flip(img, 0)
    if random.random() > 0.5:
        angle = random.uniform(-12, 12)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    return img


def stylized_augment(img):
    """Apply stack of stylized augmentations."""
    img = random_geometric(img)
    img = random_hue_sat_shift(img)
    img = random_gamma(img)
    img = random_brightness_contrast(img)
    img = random_motion_blur(img)
    img = vignette(img)
    img = random_jpeg_compression(img)
    return img


# ─── MAIN ─────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("v3-FINAL — STYLIZED RE-AUGMENTATION")
    print("=" * 60)
    print(f"Train patients: {TRAIN_PATIENTS}")
    print(f"Variants per image: {N_VARIANTS}")
    print(f"Output: {OUTPUT_DIR}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Load annotations
    df = pd.read_csv(ANNOTATIONS_CSV)
    df.columns = df.columns.str.strip()
    df['patient_id'] = df['HLY'].apply(extract_patient_id)
    train_df = df[df['patient_id'].isin(TRAIN_PATIENTS)].copy()
    print(f"Train images to augment: {len(train_df)}")

    # Locate images
    image_index = {}
    search_root = '/kaggle/input' if IS_KAGGLE else str(Path(ORIG_DATA_DIR).parent)
    for root, _, files in os.walk(search_root):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_index[f.lower()] = os.path.join(root, f)
    print(f"Indexed {len(image_index)} images on disk")

    manifest_rows = []
    skipped = 0
    
    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Augmenting"):
        fname = str(row['HLY']).strip()
        path = image_index.get(fname.lower())
        if path is None:
            skipped += 1
            continue
        
        img = cv2.imread(path)
        if img is None:
            skipped += 1
            continue
        
        # Resize to manageable size for augmentation
        h, w = img.shape[:2]
        if max(h, w) > 768:
            s = 768 / max(h, w)
            img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
        
        base_name = os.path.splitext(fname)[0]
        tissue = str(row.get('tissue type', 'unknown'))
        
        # Save original first (so manifest includes it)
        orig_out = os.path.join(OUTPUT_DIR, f"{base_name}_orig.png")
        cv2.imwrite(orig_out, img)
        manifest_rows.append({
            'HLY': f"{base_name}_orig.png",
            'tissue type': tissue,
            'patient_id': row['patient_id'],
            'is_augmented': False,
            'aug_mode': 'orig',
            'full_path': orig_out,
        })
        
        # Generate variants
        for v in range(N_VARIANTS):
            aug_img = stylized_augment(img.copy())
            out_name = f"{base_name}_aug{v}.png"
            out_path = os.path.join(OUTPUT_DIR, out_name)
            cv2.imwrite(out_path, aug_img)
            manifest_rows.append({
                'HLY': out_name,
                'tissue type': tissue,
                'patient_id': row['patient_id'],
                'is_augmented': True,
                'aug_mode': f'stylized_v{v}',
                'full_path': out_path,
            })

    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(MANIFEST_OUT, index=False)
    
    print(f"✓ Generated {len(manifest)} total images ({skipped} skipped)")
    print(f"✓ Manifest saved: {MANIFEST_OUT}")
    
    # Per-class breakdown
    print("Per-class counts:")
    for tt, cnt in manifest['tissue type'].value_counts().items():
        print(f"  {tt}: {cnt}")


if __name__ == '__main__':
    main()

v3-FINAL — STYLIZED RE-AUGMENTATION
Train patients: [4, 5, 7, 8, 10, 13, 14, 16, 21, 22, 23, 24, 25]
Variants per image: 8
Output: /kaggle/working/v3_augmented
Train images to augment: 1211
Indexed 3472 images on disk


Augmenting:   0%|          | 0/1211 [00:00<?, ?it/s]

✓ Generated 10629 total images (30 skipped)
✓ Manifest saved: /kaggle/working/v3_augmented_manifest.csv
Per-class counts:
  LGC: 4320
  HGC: 3321
  NST: 1971
  NTL: 1017


In [2]:
"""
v3-Final: Last-Shot Architecture
================================
- FROZEN DINOv2 + DenseNet121 (no fine-tuning)
- Tiny CLAM (hidden=256, single head)
- Teacher trained on train + heavy stylized augmentation
- 5 students with KL-distillation against teacher's val predictions
- 8-view image-space TTA at inference
- Constrained threshold tuning (HGC recall >= 92%)

Split B: Train=[4,5,7,8,10,13,14,16,21,22,23,24,25]
         Val=[0,1,2,9,12]
         Test=[6,11,17,18]

Prerequisite: Run augment_train_v3.py first to generate /kaggle/working/v3_augmented/
"""

import os
import re
import sys
import copy
import math
import json
import time
import random
import hashlib
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, recall_score, precision_score, f1_score
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════
# 1: CONFIGURATION
# ════════════════════════════════════════════════════════════
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    # Augmented data from augment_train_v3.py
    AUG_TRAIN_DIR = '/kaggle/working/v3_augmented'
    AUG_TRAIN_MANIFEST = '/kaggle/working/v3_augmented_manifest.csv'
    
    # Original data (for val/test)
    DATASET_ROOT = '/kaggle/input/ebt-22'
    for root, dirs, _ in os.walk('/kaggle/input'):
        if 'EndoscopicBladderTissue' in dirs:
            DATASET_ROOT = root
            break
    ORIG_DATA_DIR = os.path.join(DATASET_ROOT, 'EndoscopicBladderTissue')
    ANNOTATIONS_CSV = None
    for root, _, files in os.walk('/kaggle/input'):
        if 'annotations_fixed.csv' in files:
            ANNOTATIONS_CSV = os.path.join(root, 'annotations_fixed.csv')
            break
    
    OUTPUT_DIR = '/kaggle/working/output_v3_final'
    CACHE_DIR = '/kaggle/working/feat_cache_v3'
else:
    script_dir = Path(__file__).resolve().parent
    workspace = script_dir.parent
    AUG_TRAIN_DIR = str(workspace / 'v3_augmented')
    AUG_TRAIN_MANIFEST = str(workspace / 'v3_augmented_manifest.csv')
    ORIG_DATA_DIR = str(workspace / 'EndoscopicBladderTissue')
    ANNOTATIONS_CSV = str(workspace / 'EndoscopicBladderTissue' / 'annotations_fixed.csv')
    OUTPUT_DIR = str(workspace / 'output_v3_final')
    CACHE_DIR = str(workspace / 'feat_cache_v3')

# Splits
TRAIN_PATIENTS = [4, 5, 7, 8, 10, 13, 14, 16, 21, 22, 23, 24, 25]
VAL_PATIENTS   = [0, 1, 2, 9, 12]
TEST_PATIENTS  = [6, 11, 17, 18]

# Class config
NUM_CLASSES = 3
CLASS_NAMES = ['HGC', 'LGC', 'Normal']
LABEL_MAP = {'HGC': 'HGC', 'LGC': 'LGC', 'NST': 'Normal', 'NTL': 'Normal'}
CLASS_TO_IDX = {'HGC': 0, 'LGC': 1, 'Normal': 2}
IDX_TO_CLASS = {0: 'HGC', 1: 'LGC', 2: 'Normal'}
CANCER_CLASSES = {0, 1}

# Image preprocessing
IMAGE_RESIZE = 512
PATCH_SCALES = [96, 128, 192]
PATCH_OUTPUT_SIZE = 224
PATCH_STRIDE_FRAC = 0.5
MIN_TISSUE = 0.40
MAX_BRIGHT = 245
MIN_BRIGHT = 15
MIN_SAT = 10
MIN_FOCUS = 8.0
TOP_QUALITY_FRAC = 0.85
MAX_PATCHES_PER_IMAGE = 60
CLAHE_CLIP = 1.5
CLAHE_GRID = (16, 16)

# Feature extraction
FEAT_BATCH = 128
PATCH_BATCH_TARGET = 512
CACHE_VERSION = 'v3_final'

# CLAM (TINY — anti-overfitting)
MIL_HIDDEN = 256              # ↓ from 512
MIL_DROPOUT_TEACHER = 0.20
MIL_DROPOUT_STUDENTS = [0.30, 0.35, 0.40, 0.45, 0.50]
N_ATT_HEADS = 1               # ↓ from 4
CLAM_K_SAMPLE = 8
FEAT_NOISE_STD = 0.015
FEAT_DROP_P = 0.10

# Training
LR = 1e-4
WD = 1e-4
TEACHER_EPOCHS = 30
STUDENT_EPOCHS = 25
WARMUP_EPOCHS = 3
PATIENCE = 8
GRAD_CLIP = 1.0

# Loss
FOCAL_GAMMA = 2.0
LABEL_SMOOTH = 0.05
BAG_LOSS_W = 1.0
INST_LOSS_W = 0.05
HIER_LOSS_W = 0.10
ORDINAL_LOSS_W = 0.10
DISTILL_W = 0.30              # KL-distillation weight (students)
DISTILL_T = 4.0               # Temperature for distillation

# Class weighting
HGC_WEIGHT_BOOST = 2.5

# Patch limits
MAX_PATCHES_TRAIN = 200
MAX_PATCHES_TEST = 400

# Ensemble
N_STUDENTS = 5
STUDENT_SEEDS = [42, 7, 123, 2024, 999]

# Image-space TTA
N_TTA_VIEWS = 8

# Per-bag standardization (proven safe in v2)
USE_PER_BAG_STD = True

# Device / GPU compatibility
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CUDA_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
CUDA_CAPABILITY = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
IS_T4 = torch.cuda.is_available() and ('T4' in CUDA_NAME.upper() or CUDA_CAPABILITY == (7, 5))
IS_P100 = torch.cuda.is_available() and ('P100' in CUDA_NAME.upper() or CUDA_CAPABILITY[0] == 6)
AMP_DTYPE = torch.float16
USE_AMP = torch.cuda.is_available()

# Kaggle T4 has 16 GB memory; keep DINOv2 + DenseNet batches comfortably sized.
if IS_T4:
    FEAT_BATCH = min(FEAT_BATCH, 64)
    PATCH_BATCH_TARGET = min(PATCH_BATCH_TARGET, 384)
    CACHE_VERSION = f'{CACHE_VERSION}_t4_dino_dense121'

# P100 fallback: DINOv2/xFormers can hit unsupported CUDA kernels on sm_60.
if IS_P100:
    FEAT_BATCH = min(FEAT_BATCH, 32)
    PATCH_BATCH_TARGET = min(PATCH_BATCH_TARGET, 256)
    MAX_PATCHES_TEST = min(MAX_PATCHES_TEST, 300)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def amp_autocast():
    return torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP)


def make_grad_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# ════════════════════════════════════════════════════════════
# 2: DATA LOADING
# ════════════════════════════════════════════════════════════
def extract_patient_id(filename):
    s = str(filename)
    # Strip aug suffixes first
    s = re.sub(r'_aug\d+\.png$', '', s)
    s = re.sub(r'_orig\.png$', '', s)
    for pat in [r'case_(\d+)', r'cys_case_(\d+)']:
        m = re.search(pat, s)
        if m:
            return int(m.group(1))
    return -1


IMAGE_PATH_INDEX = {}


def scan_for_images():
    global IMAGE_PATH_INDEX
    print("" + "=" * 60)
    print("SCANNING FILESYSTEM")
    print("=" * 60)
    
    search_dirs = []
    if IS_KAGGLE:
        search_dirs = ['/kaggle/input', '/kaggle/working']
    else:
        search_dirs = [str(Path(ORIG_DATA_DIR).parent)]
    
    count = 0
    for d in search_dirs:
        if not os.path.exists(d):
            continue
        for root, _, files in os.walk(d):
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    IMAGE_PATH_INDEX[f.lower()] = os.path.join(root, f)
                    count += 1
    print(f"  Indexed {count} images")


def load_data():
    scan_for_images()
    
    print("" + "=" * 60)
    print("LOADING DATA — v3 FINAL")
    print("=" * 60)
    print(f"  Train patients: {TRAIN_PATIENTS}")
    print(f"  Val patients:   {VAL_PATIENTS}")
    print(f"  Test patients:  {TEST_PATIENTS}")
    
    # Train: load from v3 augmented manifest
    if not os.path.exists(AUG_TRAIN_MANIFEST):
        raise FileNotFoundError(f"v3 augmented manifest not found at {AUG_TRAIN_MANIFEST}."f"Run augment_train_v3.py FIRST to generate it.")
    train_df = pd.read_csv(AUG_TRAIN_MANIFEST)
    train_df['label'] = train_df['tissue type'].map(LABEL_MAP)
    train_df['target'] = train_df['label'].map(CLASS_TO_IDX)
    train_df['patient'] = train_df['patient_id']
    train_df['path'] = train_df['full_path']
    print(f"  Loaded {len(train_df)} train images from v3 augmented manifest")
    
    # Val/Test: load from original annotations
    df_orig = pd.read_csv(ANNOTATIONS_CSV)
    df_orig.columns = df_orig.columns.str.strip()
    df_orig['patient_id'] = df_orig['HLY'].apply(extract_patient_id)
    df_orig['label'] = df_orig['tissue type'].map(LABEL_MAP)
    df_orig['target'] = df_orig['label'].map(CLASS_TO_IDX)
    df_orig['patient'] = df_orig['patient_id']
    df_orig['is_augmented'] = False
    df_orig['aug_mode'] = 'orig'
    
    def resolve_orig(row):
        fname = str(row['HLY']).strip()
        return IMAGE_PATH_INDEX.get(fname.lower())
    
    df_orig['path'] = df_orig.apply(resolve_orig, axis=1)
    df_orig = df_orig[df_orig['path'].notna()].copy()
    
    val_all = df_orig[df_orig['patient_id'].isin(VAL_PATIENTS)].copy()
    val_df = val_all[val_all['imaging type'].astype(str).str.upper().eq('WLI')].copy()
    print(f"  Val WLI-only filter: kept {len(val_df)} / {len(val_all)} images; dropped {len(val_all) - len(val_df)} NBI/non-WLI images")
    test_df = df_orig[df_orig['patient_id'].isin(TEST_PATIENTS)].copy()
    
    # Verify disjointness
    train_pids = set(train_df['patient_id'].unique())
    val_pids = set(val_df['patient_id'].unique())
    test_pids = set(test_df['patient_id'].unique())
    assert len(train_pids & val_pids) == 0
    assert len(train_pids & test_pids) == 0
    assert len(val_pids & test_pids) == 0
    print("  ✓ Patient disjointness verified")
    
    for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        dist = df['label'].value_counts().to_dict()
        print(f"{name}: {len(df)} images")
        for cls in CLASS_NAMES:
            cnt = dist.get(cls, 0)
            pct = 100 * cnt / max(len(df), 1)
            print(f"    {cls}: {cnt} ({pct:.1f}%)")
    
    return train_df, val_df, test_df


# ════════════════════════════════════════════════════════════
# 3: IMAGE PREPROCESSING + PATCH EXTRACTION
# ════════════════════════════════════════════════════════════
class LabNormalizer:
    def __init__(self):
        self.ref = None

    def fit(self, images_bgr):
        stats = {'L': [], 'a': [], 'b': []}
        for img in images_bgr:
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            for i, ch in enumerate(['L', 'a', 'b']):
                stats[ch].append({'m': lab[:, :, i].mean(), 's': lab[:, :, i].std() + 1e-6})
        self.ref = {ch: {'m': np.median([s['m'] for s in stats[ch]]),
                         's': np.median([s['s'] for s in stats[ch]])} for ch in ['L', 'a', 'b']}
        return self

    def transform(self, img_bgr):
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        for i, ch in enumerate(['L', 'a', 'b']):
            c = lab[:, :, i]
            sm, ss = c.mean(), c.std() + 1e-6
            lab[:, :, i] = np.clip((c - sm) * (self.ref[ch]['s'] / ss) + self.ref[ch]['m'], 0, 255)
        lab = lab.astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def load_image(path, norm=None):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    h, w = img.shape[:2]
    s = IMAGE_RESIZE / max(h, w)
    if s != 1:
        img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    if norm:
        img = norm.transform(img)
    return img


def compute_quality(patch_bgr):
    hsv = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2HSV)
    v = hsv[:, :, 2].astype(np.float32)
    s = hsv[:, :, 1].astype(np.float32)
    mask = (v < MAX_BRIGHT) & (v > MIN_BRIGHT) & (s > MIN_SAT)
    tissue_frac = mask.sum() / mask.size
    if tissue_frac < MIN_TISSUE:
        return -1.0
    gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
    focus = cv2.Laplacian(gray, cv2.CV_64F).var()
    if focus < MIN_FOCUS:
        return -1.0
    focus_norm = min(focus / 100.0, 1.0)
    sat_std = s[mask].std() / 50.0 if mask.sum() > 10 else 0
    sat_norm = min(sat_std, 1.0)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = min(edges.sum() / (255.0 * edges.size) * 10, 1.0)
    return 0.3 * tissue_frac + 0.3 * focus_norm + 0.2 * sat_norm + 0.2 * edge_density


def extract_multiscale_patches(image_bgr, max_patches=None):
    if max_patches is None:
        max_patches = MAX_PATCHES_PER_IMAGE
    H, W = image_bgr.shape[:2]
    candidates = []
    cap = max_patches * 3
    for scale in PATCH_SCALES:
        if scale > min(H, W):
            continue
        stride = max(1, int(scale * PATCH_STRIDE_FRAC))
        for y in range(0, H - scale + 1, stride):
            for x in range(0, W - scale + 1, stride):
                if len(candidates) >= cap:
                    break
                crop = image_bgr[y:y + scale, x:x + scale]
                q = compute_quality(crop)
                if q > 0:
                    resized = cv2.resize(crop, (PATCH_OUTPUT_SIZE, PATCH_OUTPUT_SIZE), interpolation=cv2.INTER_AREA)
                    candidates.append((resized, q, scale))
            if len(candidates) >= cap:
                break
    if not candidates:
        min_dim = min(H, W)
        y0, x0 = (H - min_dim) // 2, (W - min_dim) // 2
        crop = image_bgr[y0:y0 + min_dim, x0:x0 + min_dim]
        return [cv2.resize(crop, (PATCH_OUTPUT_SIZE, PATCH_OUTPUT_SIZE))]
    candidates.sort(key=lambda x: x[1], reverse=True)
    n_keep = max(1, int(len(candidates) * TOP_QUALITY_FRAC))
    candidates = candidates[:n_keep][:max_patches]
    return [c[0] for c in candidates]


def fit_normalizer(df):
    print("  Fitting LAB normalizer...")
    samples = []
    sample_paths = df.sample(min(50, len(df))).path.values
    for fp in sample_paths:
        try:
            img = cv2.imread(fp)
            if img is not None:
                h, w = img.shape[:2]
                s = IMAGE_RESIZE / max(h, w)
                if s != 1:
                    img = cv2.resize(img, (int(w * s), int(h * s)))
                samples.append(img)
        except Exception:
            pass
    if not samples:
        return None
    norm = LabNormalizer().fit(samples)
    print(f"  ✓ Normalizer fitted on {len(samples)} images")
    return norm


# ════════════════════════════════════════════════════════════
# 4: FEATURE EXTRACTION (FROZEN BACKBONES)
# ════════════════════════════════════════════════════════════
dino_model = None
dense_model = None
feat_dim = 0


def load_backbones():
    global dino_model, dense_model, feat_dim, IMNET_MEAN, IMNET_STD
    print("" + "=" * 60)
    print("LOADING FROZEN BACKBONES")
    print("=" * 60)
    
    IMNET_MEAN = IMNET_MEAN.to(DEVICE)
    IMNET_STD = IMNET_STD.to(DEVICE)
    
    dino_dim = 0
    print("  Loading DINOv2...")
    try:
        dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        dino_model.eval().to(DEVICE)
        for p in dino_model.parameters():
            p.requires_grad = False
        dino_dim = 768
        print(f"  ✓ dinov2_vitb14 — FROZEN, dim={dino_dim}")
    except Exception as e:
        print(f"  ⚠ DINOv2 failed: {e}")
    
    densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    dense_dim = densenet.classifier.in_features
    densenet.classifier = nn.Identity()
    dense_model = densenet.eval().to(DEVICE)
    for p in dense_model.parameters():
        p.requires_grad = False
    print(f"  ✓ DenseNet121 — FROZEN, dim={dense_dim}")
    
    feat_dim = (dino_dim if dino_model else 0) + dense_dim
    print(f"  Total feature dim: {feat_dim}")
    return feat_dim


def bgr_to_tensor(patch_bgr):
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0


def _get_cache_key(path, tta_idx=None):
    parts = f"{path}|{IMAGE_RESIZE}|{PATCH_SCALES}|{MAX_PATCHES_PER_IMAGE}|{CACHE_VERSION}"
    if tta_idx is not None:
        parts += f"|tta{tta_idx}"
    return hashlib.md5(parts.encode()).hexdigest()


@torch.inference_mode()
def extract_dual_features(tensor_list):
    if not tensor_list:
        return torch.empty((0, feat_dim), dtype=torch.float16)
    all_feats = []
    for i in range(0, len(tensor_list), FEAT_BATCH):
        batch = torch.stack(tensor_list[i:i + FEAT_BATCH]).to(DEVICE, non_blocking=True)
        batch_norm = (batch - IMNET_MEAN) / IMNET_STD
        parts = []
        with amp_autocast():
            if dino_model is not None:
                dino_out = dino_model(batch_norm)
                if isinstance(dino_out, dict):
                    dino_feats = dino_out.get('x_norm_clstoken', next(iter(dino_out.values())))
                else:
                    dino_feats = dino_out
                if dino_feats.dim() > 2:
                    dino_feats = dino_feats[:, 0, :]
                parts.append(dino_feats.float().cpu())
            parts.append(dense_model(batch_norm).float().cpu())
        all_feats.append(torch.cat(parts, dim=1))
    return torch.cat(all_feats, 0).half()


def extract_features_for_split(df, desc="Extracting", norm=None, use_cache=True):
    os.makedirs(CACHE_DIR, exist_ok=True)
    results, skipped, cache_hits = [], 0, 0
    pending_tensors, pending_meta = [], []
    
    def flush():
        nonlocal pending_tensors, pending_meta
        if not pending_tensors:
            return
        feats = extract_dual_features(pending_tensors)
        start = 0
        for meta in pending_meta:
            n = meta['n_patches']
            block = feats[start:start + n]
            start += n
            if meta['cache_path']:
                try:
                    torch.save({'features': block}, meta['cache_path'])
                except Exception:
                    pass
            results.append({
                'features': block,
                'label': meta['label'],
                'label_name': meta['label_name'],
                'patient': meta['patient'],
                'path': meta['path'],
                'n_patches': block.shape[0],
            })
        pending_tensors.clear()
        pending_meta.clear()
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        cache_path = None
        if use_cache:
            cache_key = _get_cache_key(str(row.path))
            cache_path = os.path.join(CACHE_DIR, f"{cache_key}.pt")
            if os.path.exists(cache_path):
                try:
                    cached = torch.load(cache_path, map_location='cpu', weights_only=False)
                    results.append({
                        'features': cached['features'],
                        'label': int(row.target),
                        'label_name': row.label,
                        'patient': int(row.patient),
                        'path': row.path,
                        'n_patches': cached['features'].shape[0],
                    })
                    cache_hits += 1
                    continue
                except Exception:
                    pass
        
        try:
            img = load_image(str(row.path), norm)
        except Exception:
            skipped += 1
            continue
        
        patches = extract_multiscale_patches(img)
        if not patches:
            skipped += 1
            continue
        
        tensors = [bgr_to_tensor(p) for p in patches]
        if len(tensors) > MAX_PATCHES_PER_IMAGE:
            idx = random.sample(range(len(tensors)), MAX_PATCHES_PER_IMAGE)
            tensors = [tensors[i] for i in sorted(idx)]
        
        pending_tensors.extend(tensors)
        pending_meta.append({
            'label': int(row.target),
            'label_name': row.label,
            'patient': int(row.patient),
            'path': str(row.path),
            'n_patches': len(tensors),
            'cache_path': cache_path,
        })
        
        if len(pending_tensors) >= PATCH_BATCH_TARGET:
            flush()
    
    flush()
    
    if skipped:
        print(f"  ⚠ Skipped {skipped}")
    if cache_hits:
        print(f"  ⚡ Cache hits: {cache_hits}/{len(df)}")
    print(f"  ✓ Extracted {len(results)} bags")
    return results


# ════════════════════════════════════════════════════════════
# 5: TINY CLAM MODEL
# ════════════════════════════════════════════════════════════
class TinyCLAM(nn.Module):
    """
    Minimal CLAM: hidden=256, single attention head, no adapter.
    Designed to NOT memorize on N=22 patients.
    """
    def __init__(self, feat_dim_in, hidden=MIL_HIDDEN, n_classes=NUM_CLASSES,
                 dropout=0.25, k_sample=CLAM_K_SAMPLE):
        super().__init__()
        self.n_classes = n_classes
        self.k_sample = k_sample
        self.feat_noise = FEAT_NOISE_STD
        self.feat_drop = nn.Dropout(FEAT_DROP_P)
        
        # Single linear encoder (no adapter)
        self.fc = nn.Sequential(
            nn.Linear(feat_dim_in, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        # Single gated attention head per class
        self.att_branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden // 4),
                nn.Tanh(),
            ) for _ in range(n_classes)
        ])
        self.gate_branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden // 4),
                nn.Sigmoid(),
            ) for _ in range(n_classes)
        ])
        self.att_combine = nn.ModuleList([
            nn.Linear(hidden // 4, 1) for _ in range(n_classes)
        ])
        
        # Per-class instance classifiers
        self.inst_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, 64), nn.GELU(),
                nn.Linear(64, 2)
            ) for _ in range(n_classes)
        ])
        
        # Per-class bag classifiers
        self.bag_classifiers = nn.ModuleList([
            nn.Linear(hidden, 1) for _ in range(n_classes)
        ])

    def _inst_loss(self, scores, h, classifier, k):
        N = scores.shape[0]
        k = min(k, N // 2, 8)
        if k < 1:
            return torch.tensor(0.0, device=h.device)
        top_idx = torch.topk(scores, k).indices
        bot_idx = torch.topk(scores, k, largest=False).indices
        feats = torch.cat([h[top_idx], h[bot_idx]], dim=0)
        labels = torch.cat([torch.ones(k, dtype=torch.long), torch.zeros(k, dtype=torch.long)]).to(h.device)
        return F.cross_entropy(classifier(feats), labels)

    def forward(self, x, label=None):
        input_dtype = x.dtype
        
        # Per-bag standardization
        if USE_PER_BAG_STD:
            x_f = x.float()
            x = ((x_f - x_f.mean(0, keepdim=True)) / (x_f.std(0, keepdim=True) + 1e-6)).to(input_dtype)
        
        if self.training:
            x = x.float()
            x = x + torch.randn_like(x) * self.feat_noise
            x = x.to(input_dtype)
            x = self.feat_drop(x)
        
        h = self.fc(x)  # (N, hidden)
        
        logits = []
        total_inst = torch.tensor(0.0, device=x.device)
        
        for c in range(self.n_classes):
            a = self.att_branches[c](h)
            g = self.gate_branches[c](h)
            scores = self.att_combine[c](a * g).squeeze(-1)
            weights = F.softmax(scores, dim=0)
            bag = torch.sum(weights.unsqueeze(-1) * h, dim=0)
            logits.append(self.bag_classifiers[c](bag))
            
            if self.training and label is not None and label.item() == c:
                total_inst += self._inst_loss(scores.detach(), h, self.inst_classifiers[c], self.k_sample)
        
        return {'logits': torch.cat(logits), 'inst_loss': total_inst}


# ════════════════════════════════════════════════════════════
# 6: LOSSES
# ════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


def hierarchical_loss(logits, label):
    cancer_score = logits[0] + logits[1]
    normal_score = logits[2]
    binary_logit = torch.stack([cancer_score, normal_score]).unsqueeze(0)
    binary_target = torch.tensor([0] if label.item() in CANCER_CLASSES else [1],
                                 dtype=torch.long, device=label.device)
    return F.cross_entropy(binary_logit, binary_target)


def ordinal_loss(logits, label):
    probs = F.softmax(logits, dim=0)
    severity = torch.arange(NUM_CLASSES, dtype=torch.float32, device=logits.device)
    pred_sev = (probs * severity).sum()
    true_sev = severity[label]
    return (pred_sev - true_sev) ** 2


def supervised_loss(output, label, class_weights=None, focal_criterion=None):
    logits = output['logits'].unsqueeze(0)
    target = label.unsqueeze(0)
    bag_loss = focal_criterion(logits, target) if focal_criterion else \
        F.cross_entropy(logits, target, weight=class_weights, label_smoothing=LABEL_SMOOTH)
    hier = hierarchical_loss(output['logits'], label)
    ordi = ordinal_loss(output['logits'], label)
    return BAG_LOSS_W * bag_loss + HIER_LOSS_W * hier + ORDINAL_LOSS_W * ordi + INST_LOSS_W * output['inst_loss']


def kl_distill_loss(student_logits, teacher_logits, T=DISTILL_T):
    """KL divergence between student and teacher (softened)."""
    student_log_probs = F.log_softmax(student_logits / T, dim=0)
    teacher_probs = F.softmax(teacher_logits / T, dim=0)
    return F.kl_div(student_log_probs.unsqueeze(0), teacher_probs.unsqueeze(0),
                    reduction='batchmean') * (T * T)


# ════════════════════════════════════════════════════════════
# 7: TRAINING UTILITIES
# ════════════════════════════════════════════════════════════
def compute_class_weights(data_list):
    counts = Counter(d['label'] for d in data_list)
    total = sum(counts.values())
    weights = []
    for c in range(NUM_CLASSES):
        w = total / (NUM_CLASSES * max(counts.get(c, 0), 1))
        if c == CLASS_TO_IDX['HGC']:
            w *= HGC_WEIGHT_BOOST
        weights.append(w)
    wt = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f"Class weights (HGC ×{HGC_WEIGHT_BOOST}):")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"    {cls}: count={counts.get(i,0)} → w={weights[i]:.3f}")
    return wt


def class_balanced_sample(data_list):
    by_class = defaultdict(list)
    for d in data_list:
        by_class[d['label']].append(d)
    max_count = max(len(v) for v in by_class.values())
    balanced = []
    for cls, items in by_class.items():
        balanced.extend(items)
        if len(items) < max_count:
            balanced.extend(random.choices(items, k=max_count - len(items)))
    random.shuffle(balanced)
    return balanced


def warmup_cosine(optimizer, warmup, total):
    def lr_lambda(epoch):
        if epoch < warmup:
            return (epoch + 1) / warmup
        progress = (epoch - warmup) / max(total - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ════════════════════════════════════════════════════════════
# 8: TEACHER TRAINING
# ════════════════════════════════════════════════════════════
def train_teacher(train_data, val_data, class_weights):
    print("" + "=" * 60)
    print("TRAINING TEACHER")
    print("=" * 60)
    
    set_seed(42)
    teacher = TinyCLAM(
        feat_dim_in=feat_dim,
        hidden=MIL_HIDDEN,
        dropout=MIL_DROPOUT_TEACHER,
    ).to(DEVICE)
    
    n_params = sum(p.numel() for p in teacher.parameters() if p.requires_grad)
    print(f"  Teacher params: {n_params:,}")
    
    focal = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights, label_smoothing=LABEL_SMOOTH).to(DEVICE)
    optim = torch.optim.AdamW(teacher.parameters(), lr=LR, weight_decay=WD)
    sched = warmup_cosine(optim, WARMUP_EPOCHS, TEACHER_EPOCHS)
    scaler = make_grad_scaler()
    
    best_val_loss = float('inf')
    best_state = None
    patience = 0
    
    for epoch in range(TEACHER_EPOCHS):
        teacher.train()
        epoch_data = class_balanced_sample(train_data)
        
        running = 0.0
        correct = 0
        total = 0
        
        for sample in epoch_data:
            feats = sample['features'].to(DEVICE)
            if feats.shape[0] > MAX_PATCHES_TRAIN:
                idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TRAIN]
                feats = feats[idx]
            label = torch.tensor(sample['label'], dtype=torch.long, device=DEVICE)
            
            optim.zero_grad(set_to_none=True)
            with amp_autocast():
                out = teacher(feats, label=label)
                loss = supervised_loss(out, label, class_weights, focal)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            nn.utils.clip_grad_norm_(teacher.parameters(), GRAD_CLIP)
            scaler.step(optim)
            scaler.update()
            
            running += loss.item()
            correct += (out['logits'].argmax().item() == sample['label'])
            total += 1
        
        sched.step()
        train_loss = running / max(total, 1)
        train_acc = correct / max(total, 1)
        
        # Val
        teacher.eval()
        v_loss = 0.0
        v_correct = 0
        with torch.no_grad():
            for sample in val_data:
                feats = sample['features'].to(DEVICE)
                if feats.shape[0] > MAX_PATCHES_TEST:
                    idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
                    feats = feats[idx]
                label = torch.tensor(sample['label'], dtype=torch.long, device=DEVICE)
                with amp_autocast():
                    out = teacher(feats)
                loss = F.cross_entropy(out['logits'].float().unsqueeze(0),
                                       label.unsqueeze(0), weight=class_weights)
                v_loss += loss.item()
                if out['logits'].argmax().item() == sample['label']:
                    v_correct += 1
        v_loss /= len(val_data)
        v_acc = v_correct / len(val_data)
        
        improved = ""
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state = copy.deepcopy(teacher.state_dict())
            patience = 0
            improved = " *"
        else:
            patience += 1
        
        if epoch < 3 or (epoch + 1) % 3 == 0 or improved or epoch == TEACHER_EPOCHS - 1:
            print(f"    Epoch {epoch+1:3d}: train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
                  f"val_loss={v_loss:.4f} val_acc={v_acc:.3f}{improved}")
        
        if patience >= PATIENCE:
            print(f"    Early stop at epoch {epoch+1}")
            break
    
    if best_state:
        teacher.load_state_dict(best_state)
        print(f"    ✓ Restored teacher (val_loss={best_val_loss:.4f})")
    
    return teacher


# ════════════════════════════════════════════════════════════
# 9: STUDENT TRAINING (with teacher distillation on val)
# ════════════════════════════════════════════════════════════
@torch.no_grad()
def compute_teacher_val_logits(teacher, val_data):
    """Pre-compute teacher's logits on val for distillation targets."""
    teacher.eval()
    teacher_logits = []
    for sample in val_data:
        feats = sample['features'].to(DEVICE)
        if feats.shape[0] > MAX_PATCHES_TEST:
            idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
            feats = feats[idx]
        with amp_autocast():
            out = teacher(feats)
        teacher_logits.append(out['logits'].float().detach().cpu())
    return teacher_logits


def train_student(student_idx, train_data, val_data, val_teacher_logits,
                  class_weights):
    seed = STUDENT_SEEDS[student_idx]
    dropout = MIL_DROPOUT_STUDENTS[student_idx]
    
    print(f"── Student {student_idx+1}/{N_STUDENTS}: seed={seed}, dropout={dropout} ──")
    set_seed(seed)
    
    student = TinyCLAM(
        feat_dim_in=feat_dim,
        hidden=MIL_HIDDEN,
        dropout=dropout,
    ).to(DEVICE)
    
    focal = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights, label_smoothing=LABEL_SMOOTH).to(DEVICE)
    optim = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WD)
    sched = warmup_cosine(optim, WARMUP_EPOCHS, STUDENT_EPOCHS)
    scaler = make_grad_scaler()
    
    best_val_loss = float('inf')
    best_state = None
    patience = 0
    
    for epoch in range(STUDENT_EPOCHS):
        student.train()
        epoch_data = class_balanced_sample(train_data)
        
        # Mix in val samples (with teacher logits) for distillation
        # We iterate train data with supervised loss + sample val for distill
        running_sup = 0.0
        running_distill = 0.0
        correct = 0
        total = 0
        
        for sample in epoch_data:
            feats = sample['features'].to(DEVICE)
            if feats.shape[0] > MAX_PATCHES_TRAIN:
                idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TRAIN]
                feats = feats[idx]
            label = torch.tensor(sample['label'], dtype=torch.long, device=DEVICE)
            
            optim.zero_grad(set_to_none=True)
            with amp_autocast():
                out = student(feats, label=label)
                sup_loss = supervised_loss(out, label, class_weights, focal)
            
            scaler.scale(sup_loss).backward()
            running_sup += sup_loss.item()
            
            # Distillation step: forward on a random val sample, KL against teacher
            if random.random() < 0.5:  # distill on ~half of train steps
                val_idx = random.randrange(len(val_data))
                val_sample = val_data[val_idx]
                val_feats = val_sample['features'].to(DEVICE)
                if val_feats.shape[0] > MAX_PATCHES_TRAIN:
                    idx = torch.randperm(val_feats.shape[0])[:MAX_PATCHES_TRAIN]
                    val_feats = val_feats[idx]
                t_logits = val_teacher_logits[val_idx].to(DEVICE)
                
                with amp_autocast():
                    val_out = student(val_feats)
                    distill_loss = DISTILL_W * kl_distill_loss(val_out['logits'].float(), t_logits)
                
                scaler.scale(distill_loss).backward()
                running_distill += distill_loss.item()
            
            scaler.unscale_(optim)
            nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
            scaler.step(optim)
            scaler.update()
            
            correct += (out['logits'].argmax().item() == sample['label'])
            total += 1
        
        sched.step()
        train_loss = running_sup / max(total, 1)
        distill_avg = running_distill / max(total, 1)
        train_acc = correct / max(total, 1)
        
        # Val eval
        student.eval()
        v_loss = 0.0
        v_correct = 0
        with torch.no_grad():
            for sample in val_data:
                feats = sample['features'].to(DEVICE)
                if feats.shape[0] > MAX_PATCHES_TEST:
                    idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
                    feats = feats[idx]
                label = torch.tensor(sample['label'], dtype=torch.long, device=DEVICE)
                with amp_autocast():
                    out = student(feats)
                loss = F.cross_entropy(out['logits'].float().unsqueeze(0),
                                       label.unsqueeze(0), weight=class_weights)
                v_loss += loss.item()
                if out['logits'].argmax().item() == sample['label']:
                    v_correct += 1
        v_loss /= len(val_data)
        v_acc = v_correct / len(val_data)
        
        improved = ""
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state = copy.deepcopy(student.state_dict())
            patience = 0
            improved = " *"
        else:
            patience += 1
        
        if epoch < 3 or (epoch + 1) % 3 == 0 or improved or epoch == STUDENT_EPOCHS - 1:
            print(f"    Epoch {epoch+1:3d}: sup={train_loss:.4f} distill={distill_avg:.4f} "
                  f"train_acc={train_acc:.3f} | val_loss={v_loss:.4f} val_acc={v_acc:.3f}{improved}")
        
        if patience >= PATIENCE:
            print(f"    Early stop at epoch {epoch+1}")
            break
    
    if best_state:
        student.load_state_dict(best_state)
        print(f"    ✓ Restored student (val_loss={best_val_loss:.4f})")
    
    return student


# ════════════════════════════════════════════════════════════
# 10: IMAGE-SPACE TTA + INFERENCE
# ════════════════════════════════════════════════════════════
def tta_transform(img_bgr, tta_idx):
    """Apply one of 8 TTA transformations to image."""
    if tta_idx == 0:
        return img_bgr  # identity
    elif tta_idx == 1:
        return cv2.flip(img_bgr, 1)  # h-flip
    elif tta_idx == 2:
        return cv2.flip(img_bgr, 0)  # v-flip
    elif tta_idx == 3:
        return cv2.flip(cv2.flip(img_bgr, 1), 0)  # h+v
    elif tta_idx == 4:
        return cv2.rotate(img_bgr, cv2.ROTATE_90_CLOCKWISE)
    elif tta_idx == 5:
        return cv2.rotate(img_bgr, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif tta_idx == 6:
        # Slight brightness up
        return cv2.convertScaleAbs(img_bgr, alpha=1.1, beta=10)
    elif tta_idx == 7:
        # Slight brightness down
        return cv2.convertScaleAbs(img_bgr, alpha=0.9, beta=-10)
    return img_bgr


def extract_features_with_tta(df, norm, desc="TTA features"):
    """Extract features for all images × N_TTA_VIEWS, with caching."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    results = []  # results[i] = list of N_TTA_VIEWS feature tensors
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        per_image_features = []
        
        # Load image once
        try:
            img_orig = load_image(str(row.path), norm)
        except Exception:
            results.append(None)
            continue
        
        for tta_idx in range(N_TTA_VIEWS):
            cache_key = _get_cache_key(str(row.path), tta_idx=tta_idx)
            cache_path = os.path.join(CACHE_DIR, f"{cache_key}.pt")
            
            if os.path.exists(cache_path):
                try:
                    cached = torch.load(cache_path, map_location='cpu', weights_only=False)
                    per_image_features.append(cached['features'])
                    continue
                except Exception:
                    pass
            
            # Apply TTA
            img_tta = tta_transform(img_orig, tta_idx)
            patches = extract_multiscale_patches(img_tta)
            if not patches:
                per_image_features.append(None)
                continue
            tensors = [bgr_to_tensor(p) for p in patches]
            if len(tensors) > MAX_PATCHES_PER_IMAGE:
                idx = random.sample(range(len(tensors)), MAX_PATCHES_PER_IMAGE)
                tensors = [tensors[i] for i in sorted(idx)]
            
            feats = extract_dual_features(tensors)
            try:
                torch.save({'features': feats}, cache_path)
            except Exception:
                pass
            per_image_features.append(feats)
        
        results.append({
            'tta_features': per_image_features,
            'label': int(row.target),
            'patient': int(row.patient),
            'path': str(row.path),
        })
    
    print(f"  ✓ Extracted TTA features for {len(results)} images × {N_TTA_VIEWS} views")
    return results


@torch.no_grad()
def predict_with_tta_ensemble(students, tta_data_item):
    """For one image: run all students × all TTA views, average softmax."""
    if tta_data_item is None:
        return np.array([1/3, 1/3, 1/3])
    
    all_probs = []
    for tta_feats in tta_data_item['tta_features']:
        if tta_feats is None or tta_feats.shape[0] == 0:
            continue
        feats = tta_feats.to(DEVICE)
        if feats.shape[0] > MAX_PATCHES_TEST:
            idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
            feats = feats[idx]
        
        for student in students:
            student.eval()
            with amp_autocast():
                out = student(feats)
            probs = F.softmax(out['logits'].float(), dim=0).cpu().numpy()
            all_probs.append(probs)
    
    if not all_probs:
        return np.array([1/3, 1/3, 1/3])
    avg_probs = np.mean(all_probs, axis=0)
    avg_probs = avg_probs / (avg_probs.sum() + 1e-8)
    return avg_probs


def evaluate_full(students, tta_data, desc="Evaluating"):
    all_preds = []
    all_labels = []
    all_probs = []
    for item in tqdm(tta_data, desc=desc):
        if item is None:
            continue
        probs = predict_with_tta_ensemble(students, item)
        all_preds.append(int(np.argmax(probs)))
        all_labels.append(item['label'])
        all_probs.append(probs)
    return all_preds, all_labels, all_probs


# ════════════════════════════════════════════════════════════
# 11: CONSTRAINED THRESHOLD TUNING
# ════════════════════════════════════════════════════════════
def tune_constrained_thresholds(val_labels, val_probs, hgc_recall_target=0.92):
    print("" + "=" * 60)
    print(f"  CONSTRAINED THRESHOLD TUNING (HGC recall ≥ {hgc_recall_target*100:.0f}%)")
    print("=" * 60)
    
    val_labels = np.array(val_labels)
    val_probs = np.array(val_probs)
    hgc_idx = CLASS_TO_IDX['HGC']
    lgc_idx = CLASS_TO_IDX['LGC']
    hgc_true = (val_labels == hgc_idx)
    n_hgc = hgc_true.sum()
    
    base_preds = np.argmax(val_probs, axis=1)
    base_bal = balanced_accuracy_score(val_labels, base_preds)
    base_hgc_r = ((base_preds == hgc_idx) & hgc_true).sum() / max(n_hgc, 1)
    print(f"  Argmax baseline: bal_acc={base_bal*100:.1f}%, HGC_recall={base_hgc_r*100:.1f}%")
    
    best = None
    best_bal = -1
    
    for hgc_t in np.arange(0.15, 0.55, 0.025):
        for lgc_t in np.arange(0.25, 0.55, 0.025):
            preds = []
            for p in val_probs:
                if p[hgc_idx] > hgc_t:
                    preds.append(hgc_idx)
                elif p[lgc_idx] > lgc_t:
                    preds.append(lgc_idx)
                else:
                    preds.append(int(np.argmax(p)))
            preds = np.array(preds)
            hgc_r = ((preds == hgc_idx) & hgc_true).sum() / max(n_hgc, 1)
            if hgc_r < hgc_recall_target:
                continue
            bal = balanced_accuracy_score(val_labels, preds)
            if bal > best_bal:
                best_bal = bal
                hgc_p = ((preds == hgc_idx) & hgc_true).sum() / max((preds == hgc_idx).sum(), 1)
                best = {
                    'hgc_threshold': float(hgc_t),
                    'lgc_threshold': float(lgc_t),
                    'bal_acc': bal,
                    'hgc_recall': hgc_r,
                    'hgc_precision': hgc_p,
                    'accuracy': accuracy_score(val_labels, preds),
                }
    
    if best:
        print(f"  ✓ Best: HGC_t={best['hgc_threshold']:.3f}, LGC_t={best['lgc_threshold']:.3f}")
        print(f"    val: bal_acc={best['bal_acc']*100:.1f}%, "
              f"HGC_R={best['hgc_recall']*100:.1f}%, HGC_P={best['hgc_precision']*100:.1f}%")
        return {hgc_idx: best['hgc_threshold'], lgc_idx: best['lgc_threshold']}, best
    
    print(f"  ⚠ Constraint not satisfiable. Falling back to lower target...")
    if hgc_recall_target > 0.85:
        return tune_constrained_thresholds(val_labels, val_probs, hgc_recall_target=0.85)
    return {hgc_idx: 0.3, lgc_idx: 0.4}, None


def apply_thresholds(probs_list, thresholds):
    preds = []
    hgc_idx = CLASS_TO_IDX['HGC']
    lgc_idx = CLASS_TO_IDX['LGC']
    hgc_t = thresholds.get(hgc_idx, 0.5)
    lgc_t = thresholds.get(lgc_idx, 0.5)
    for p in probs_list:
        if p[hgc_idx] > hgc_t:
            preds.append(hgc_idx)
        elif p[lgc_idx] > lgc_t:
            preds.append(lgc_idx)
        else:
            preds.append(int(np.argmax(p)))
    return preds


# ════════════════════════════════════════════════════════════
# 12: REPORTING
# ════════════════════════════════════════════════════════════
def print_results(preds, labels, name):
    preds = np.array(preds)
    labels = np.array(labels)
    acc = accuracy_score(labels, preds)
    bal = balanced_accuracy_score(labels, preds)
    print(f"{'=' * 60}{name}{'=' * 60}")
    print(f"  Accuracy:          {acc*100:.2f}%")
    print(f"  Balanced Accuracy: {bal*100:.2f}%")
    print(f"{classification_report(labels, preds, target_names=CLASS_NAMES, digits=4, zero_division=0)}")
    hgc_idx = CLASS_TO_IDX['HGC']
    hgc_true = (labels == hgc_idx)
    hgc_pred = (preds == hgc_idx)
    hgc_tp = (hgc_pred & hgc_true).sum()
    hgc_r = hgc_tp / max(hgc_true.sum(), 1)
    hgc_p = hgc_tp / max(hgc_pred.sum(), 1)
    print(f"  HGC: Recall={hgc_r*100:.2f}%, Precision={hgc_p*100:.2f}%")
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    print(f"Confusion Matrix:{'':>10} {'  '.join(f'{c:>8s}' for c in CLASS_NAMES)}")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"  {cls:>10} {'  '.join(f'{cm[i,j]:8d}' for j in range(NUM_CLASSES))}")
    target_acc = "✅" if acc >= 0.90 else "❌"
    target_hgc = "✅" if hgc_r >= 0.95 else "❌"
    print(f"Target acc≥90%:    {target_acc} ({acc*100:.1f}%)")
    print(f"  Target HGC R≥95%:  {target_hgc} ({hgc_r*100:.1f}%)")
    return {'accuracy': acc, 'balanced_accuracy': bal,
            'hgc_recall': hgc_r, 'hgc_precision': hgc_p}


def save_cm(preds, labels, fname, title):
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.close()


# ════════════════════════════════════════════════════════════
# 13: MAIN
# ════════════════════════════════════════════════════════════
def main():
    start = time.time()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║  Bladder Classification — v3 FINAL (LAST SHOT)              ║")
    print("║  Frozen DINOv2 + DenseNet → Tiny CLAM (256/1head)          ║")
    print("║  Teacher → 5 Students with KL-distill on val               ║")
    print("║  8-view image-TTA + Constrained threshold tuning           ║")
    print(f"║  Stylized 8× re-augmentation (run augment_train_v3.py first) ║")
    print("╚══════════════════════════════════════════════════════════════╝")
    print(f"Device: {DEVICE} | GPU: {CUDA_NAME} | capability: {CUDA_CAPABILITY} | AMP: {USE_AMP} ({AMP_DTYPE}) | T4 mode: {IS_T4} | P100 fallback: {IS_P100}")
    
    train_df, val_df, test_df = load_data()
    
    global feat_dim
    feat_dim = load_backbones()
    
    norm = fit_normalizer(train_df.sample(min(len(train_df), 200), random_state=42))
    
    print("" + "=" * 60)
    print("EXTRACTING TRAIN FEATURES")
    print("=" * 60)
    train_data = extract_features_for_split(train_df, "Train", norm, use_cache=True)
    
    print("" + "=" * 60)
    print("EXTRACTING VAL FEATURES (single view, no TTA — for training)")
    print("=" * 60)
    val_data = extract_features_for_split(val_df, "Val (clean)", norm, use_cache=True)
    
    print(f"Train bags: {len(train_data)} | Val bags (clean): {len(val_data)}")
    
    class_weights = compute_class_weights(train_data)
    
    # ─── Train teacher ───
    teacher = train_teacher(train_data, val_data, class_weights)
    
    # ─── Compute teacher's val logits for distillation ───
    print("Computing teacher's val logits for distillation targets...")
    teacher_val_logits = compute_teacher_val_logits(teacher, val_data)
    print(f"  ✓ Cached {len(teacher_val_logits)} teacher logits")
    
    # ─── Train students ───
    print("" + "=" * 60)
    print(f"TRAINING {N_STUDENTS} STUDENTS WITH DISTILLATION")
    print("=" * 60)
    students = []
    for s_idx in range(N_STUDENTS):
        student = train_student(s_idx, train_data, val_data, teacher_val_logits, class_weights)
        student.eval()
        students.append(student)
    
    # ─── Extract TTA features for val + test ───
    print("" + "=" * 60)
    print("EXTRACTING TTA FEATURES FOR VAL + TEST")
    print("=" * 60)
    val_tta = extract_features_with_tta(val_df, norm, "Val TTA")
    test_tta = extract_features_with_tta(test_df, norm, "Test TTA")
    
    # ─── Evaluate students ensemble (argmax baseline) ───
    print("" + "=" * 60)
    print("BASELINE EVAL (ensemble + TTA, argmax)")
    print("=" * 60)
    val_preds_b, val_labels, val_probs_b = evaluate_full(students, val_tta, "Val baseline")
    print_results(val_preds_b, val_labels, "VAL — ARGMAX BASELINE")
    
    test_preds_b, test_labels, test_probs_b = evaluate_full(students, test_tta, "Test baseline")
    base_test = print_results(test_preds_b, test_labels, "TEST — ARGMAX BASELINE")
    
    # ─── Tune thresholds ───
    thresholds, val_thresh_meta = tune_constrained_thresholds(val_labels, val_probs_b)
    
    # ─── Final eval with thresholds ───
    print("" + "=" * 60)
    print("FINAL EVAL (ensemble + TTA + tuned thresholds)")
    print("=" * 60)
    val_preds_t = apply_thresholds(val_probs_b, thresholds)
    val_metrics = print_results(val_preds_t, val_labels, "VAL — FINAL")
    
    test_preds_t = apply_thresholds(test_probs_b, thresholds)
    test_metrics = print_results(test_preds_t, test_labels, "TEST — FINAL (HEADLINE)")
    
    # ─── Per-patient breakdown ───
    print("" + "=" * 60)
    print("PER-PATIENT TEST BREAKDOWN")
    print("=" * 60)
    test_pids = [item['patient'] for item in test_tta if item is not None]
    for pid in sorted(set(test_pids)):
        idxs = [i for i, p in enumerate(test_pids) if p == pid]
        p_lab = [test_labels[i] for i in idxs]
        p_prd = [test_preds_t[i] for i in idxs]
        p_acc = accuracy_score(p_lab, p_prd)
        dist = Counter(p_lab)
        dist_str = ', '.join(f"{IDX_TO_CLASS[k]}={v}" for k, v in sorted(dist.items()))
        print(f"  Patient {pid}: {len(idxs)} imgs ({dist_str}) → acc={p_acc*100:.1f}%")
    
    # ─── Improvement table ───
    print("" + "=" * 60)
    print("IMPROVEMENT (Argmax → Final)")
    print("=" * 60)
    print(f"{'Metric':<25s} {'Argmax':>10s} {'Final':>10s} {'Δ':>10s}")
    print("  " + "─" * 55)
    for name, b, f in [
        ('Accuracy', base_test['accuracy'], test_metrics['accuracy']),
        ('Balanced Acc', base_test['balanced_accuracy'], test_metrics['balanced_accuracy']),
        ('HGC Recall', base_test['hgc_recall'], test_metrics['hgc_recall']),
        ('HGC Precision', base_test['hgc_precision'], test_metrics['hgc_precision']),
    ]:
        delta = f - b
        sign = "+" if delta >= 0 else ""
        print(f"  {name:<23s} {b*100:>9.1f}% {f*100:>9.1f}% {sign}{delta*100:>9.1f}%")
    
    # ─── Save ───
    save_cm(test_preds_t, test_labels, os.path.join(OUTPUT_DIR, 'cm_test_final.png'), "v3-Final Test")
    save_cm(val_preds_t, val_labels, os.path.join(OUTPUT_DIR, 'cm_val_final.png'), "v3-Final Val")
    save_cm(test_preds_b, test_labels, os.path.join(OUTPUT_DIR, 'cm_test_baseline.png'), "v3 Argmax Test")
    
    elapsed = time.time() - start
    results = {
        'experiment': 'v3_final',
        'timestamp': datetime.now().isoformat(),
        'runtime_minutes': elapsed / 60,
        'split': {'train': TRAIN_PATIENTS, 'val': VAL_PATIENTS, 'test': TEST_PATIENTS},
        'config': {
            'mil_hidden': MIL_HIDDEN,
            'n_att_heads': N_ATT_HEADS,
            'n_students': N_STUDENTS,
            'student_dropouts': MIL_DROPOUT_STUDENTS,
            'n_tta_views': N_TTA_VIEWS,
            'distill_w': DISTILL_W,
            'distill_T': DISTILL_T,
            'hgc_weight_boost': HGC_WEIGHT_BOOST,
            'frozen_backbone': True,
            'per_bag_std': USE_PER_BAG_STD,
            'gpu_name': CUDA_NAME,
            'cuda_capability': CUDA_CAPABILITY,
            't4_mode': IS_T4,
            'p100_fallback': IS_P100,
            'feat_batch': FEAT_BATCH,
            'patch_batch_target': PATCH_BATCH_TARGET,
            'max_patches_test': MAX_PATCHES_TEST,
        },
        'baseline_test': base_test,
        'final_test': test_metrics,
        'final_val': val_metrics,
        'thresholds': {IDX_TO_CLASS[k]: v for k, v in thresholds.items()},
    }
    with open(os.path.join(OUTPUT_DIR, 'v3_final_results.json'), 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    print(f"{'═' * 60}")
    print(f"  v3-FINAL COMPLETE — Runtime: {elapsed/60:.1f} min")
    print(f"  Test: acc={test_metrics['accuracy']*100:.1f}% bal={test_metrics['balanced_accuracy']*100:.1f}% "f"HGC_R={test_metrics['hgc_recall']*100:.1f}%")
    print(f"{'═' * 60}")


if __name__ == '__main__':
    main()

╔══════════════════════════════════════════════════════════════╗
║  Bladder Classification — v3 FINAL (LAST SHOT)              ║
║  Frozen DINOv2 + DenseNet → Tiny CLAM (256/1head)          ║
║  Teacher → 5 Students with KL-distill on val               ║
║  8-view image-TTA + Constrained threshold tuning           ║
║  Stylized 8× re-augmentation (run augment_train_v3.py first) ║
╚══════════════════════════════════════════════════════════════╝
Device: cuda | GPU: Tesla T4 | capability: (7, 5) | AMP: True (torch.float16) | T4 mode: True | P100 fallback: False
SCANNING FILESYSTEM
  Indexed 14101 images
LOADING DATA — v3 FINAL
  Train patients: [4, 5, 7, 8, 10, 13, 14, 16, 21, 22, 23, 24, 25]
  Val patients:   [0, 1, 2, 9, 12]
  Test patients:  [6, 11, 17, 18]
  Loaded 10629 train images from v3 augmented manifest
  Val WLI-only filter: kept 165 / 235 images; dropped 70 NBI/non-WLI images
  ✓ Patient disjointness verified
Train: 10629 images
    HGC: 3321 (31.2%)
    LGC: 4320 (40.6%)
   

100%|██████████| 330M/330M [00:01<00:00, 185MB/s]  


  ✓ dinov2_vitb14 — FROZEN, dim=768
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 150MB/s] 


  ✓ DenseNet121 — FROZEN, dim=1024
  Total feature dim: 1792
  Fitting LAB normalizer...
  ✓ Normalizer fitted on 50 images
EXTRACTING TRAIN FEATURES


...

  ✓ Extracted TTA features for 301 images × 8 views
BASELINE EVAL (ensemble + TTA, argmax)


Val baseline:   0%|          | 0/165 [00:00<?, ?it/s]

============================================================VAL — ARGMAX BASELINE============================================================
  Accuracy:          76.97%
  Balanced Accuracy: 50.69%
              precision    recall  f1-score   support

         HGC     0.0000    0.0000    0.0000        15
         LGC     0.8571    0.5581    0.6761        43
      Normal     0.8655    0.9626    0.9115       107

    accuracy                         0.7697       165
   macro avg     0.5742    0.5069    0.5292       165
weighted avg     0.7847    0.7697    0.7673       165

  HGC: Recall=0.00%, Precision=0.00%
Confusion Matrix:                HGC       LGC    Normal
         HGC        0         0        15
         LGC       18        24         1
      Normal        0         4       103
Target acc≥90%:    ❌ (77.0%)
  Target HGC R≥95%:  ❌ (0.0%)


Test baseline:   0%|          | 0/301 [00:00<?, ?it/s]

============================================================TEST — ARGMAX BASELINE============================================================
  Accuracy:          81.40%
  Balanced Accuracy: 77.90%
              precision    recall  f1-score   support

         HGC     0.6571    0.9324    0.7709        74
         LGC     0.9000    0.4390    0.5902        82
      Normal     0.8974    0.9655    0.9302       145

    accuracy                         0.8140       301
   macro avg     0.8182    0.7790    0.7638       301
weighted avg     0.8391    0.8140    0.7984       301

  HGC: Recall=93.24%, Precision=65.71%
Confusion Matrix:                HGC       LGC    Normal
         HGC       69         2         3
         LGC       33        36        13
      Normal        3         2       140
Target acc≥90%:    ❌ (81.4%)
  Target HGC R≥95%:  ❌ (93.2%)
  CONSTRAINED THRESHOLD TUNING (HGC recall ≥ 92%)
  Argmax baseline: bal_acc=50.7%, HGC_recall=0.0%
  ⚠ Constraint not satisfiable. Fallin